# Lead Variant Effect dataset preparation

The aim of this notebook is to collect the information about the effect of the credible set lead variants.
**This includes**:

- Addition of **Major population sample size** and **size of cases/controls** from _studyIndex_,
- Addition of **VEP consequence score** derived annotations from _variantIndex_
- Addition of **study specific major ancestry variant AF** (allele frequency)<a name="out of sample AF"></a>[<sup>[1]</sup>](#cite_note-1) annotations from _variantIndex_
- Calculation of **MAF (Minor Allele Frequency)** based on AF of the **credible set lead variants** derived from _studyLocus_
- Calculation of **Variance Explained by lead variant**
- Calculation of the **Rescaled estimated effect sizes** based on the trait class (dichotomous or continuous) and the MAF of the lead variant.

<a name="cite_note-1"></a>1. [^](#cite_ref-1) AF is derived from GnomAD v4.1 allele frequencies from joint Exome and WGS datasets.


In [1]:
# Ensure proper java version < 11
!java -version


openjdk version "17.0.14" 2025-01-21
OpenJDK Runtime Environment Temurin-17.0.14+7 (build 17.0.14+7)
OpenJDK 64-Bit Server VM Temurin-17.0.14+7 (build 17.0.14+7, mixed mode, sharing)


## Session setup

- Create the sparkSession
- Set all input/output paths


In [3]:
import json
from pathlib import Path

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from pyspark.sql import functions as f


Loading BokehJS ...

/home/ss60/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [ ]:
RELEASE_PATH = "gs://ot_orchestration/gentropy_manuscript/data/26.03"


In [12]:
session = Session(extended_spark_conf={"spark.driver.memory": "16G", "spark.executor.memory": "16G"}, spark_uri="yarn")
variant_index_path = f"{RELEASE_PATH}/output/variant"
study_index_path = f"{RELEASE_PATH}/output/study"
credible_set_path = f"{RELEASE_PATH}/output/credible_set"


26/04/09 09:09:38 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:09:38 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:09:38 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:09:38 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:

In [ ]:
output_dataset_path = f"{RELEASE_PATH}/intermediate/lead_variant_effect"
output_dataset_schema = f"lead_variant_effect_schema.json"


In [14]:
session.spark


## Building temporary dataset

The temporary dataset needs to be build from the _studyIndex_, _studyLocus_ and _variantIndex_ datasets.


In [15]:
vi = VariantIndex.from_parquet(session, variant_index_path)
si = StudyIndex.from_parquet(session, study_index_path)
cs = StudyLocus.from_parquet(session, credible_set_path)


_cs = cs.df.select(
    f.col("studyId"),
    f.col("studyLocusId"),
    f.col("variantId"),
    f.col("beta"),
    f.col("zScore"),
    f.col("pValueMantissa"),
    f.col("pValueExponent"),
    f.col("standardError"),
    f.col("finemappingMethod"),
    f.col("studyType"),
    f.col("locus"),
    f.col("isTransQtl"),
)
_si = si.df.select(
    f.col("studyId"),
    f.col("nSamples"),
    f.col("nControls"),
    f.col("nCases"),
    f.col("geneId"),  # for molqtl traits
    f.col("diseaseIds"),  # for disease & measurement traits
    f.col("biosampleId"),
    f.col("traitFromSourceMappedIds"),
    f.col("ldPopulationStructure"),
    f.col("traitFromSource"),
    f.col("traitFromSourceMappedIds"),
)

_vi = vi.df.select(
    f.col("variantId"),
    f.col("allelefrequencies"),
    f.col("variantEffect"),
    f.col("transcriptConsequences"),
    f.col("chromosome"),
    f.col("position"),
    f.col("referenceAllele"),
    f.col("alternateAllele"),
)

dataset = _cs.join(_si, how="left", on="studyId").join(_vi, how="left", on="variantId")


## MAF Calculation

To add the MAF (Minor Allele Frequency) to the dataset we need to extract the major ancestry from _studyIndex_ and use it to extract the relevant allele frequency from the _variantIndex_ dataset.

The MAF is calculated as follows:

<ol>
    <li>Extract the major ancestry from the <code>studyIndex</code> dataset.</li>
    <ol>
        <li>In case there are multiple ancestries that match the <code>max(relativeSampleSize)</code>, and one of them is <code>NFE</code>, use <code>NFE</code> as the major ancestry.</li>
        <li>In case there are multiple ancestries that match the <code>max(relativeSampleSize)</code> and none of them is <code>NFE</code>, use the first ancestry in the list as the major ancestry.</li>
        <li>If there is no ancestry in the list, use <code>NFE</code> as the major ancestry, assign the <code>relativeSampleSize</code> to 0.0</li>
    </ol>
    <li>Extract the allele frequency for the major ancestry from the <code>variantIndex</code> dataset.</li>
</ol>

The output of this step contains the following columns:

- `majorLdPopulation`: defined as the major ancestry from the _studyIndex_ dataset. This column contains two fields:
  - `ldPopulation`: the major ancestry value (ex. `nfe`, `afr`, etc.). In case there is no ancestry, the value is set to `nfe`. (ex. molecular QTLs).
  - `relativeSampleSize`: derived from LdPopulationStructure field. In case there is no ancestry, the value is set to `0.0`. (ex. molecular QTLs).
- `majorLdPopulationMaf`: defined as the MAF for the major ancestry from the _variantIndex_ dataset, with two fields
  - `value` represents the MAF, while
  - `type` represents weather the MAF was calculated from major ancestry AF had to be flipped or not.
- `majorLdPopulationAf`: defined as the AF for the major ancestry from the _variantIndex_ dataset. With two fields:
  - `populationName` represents the major ancestry name (ex. `nfe`, `afr`, etc.).
  - `alleleFrequency` represents the Allele Frequency for the major ancestry.
- `ldStructure`: defined as the LD structure of the populations present in the _studyIndex_ dataset:
  - `type`: type of the LD structure derived from major ancestries, ex.
    - `only_major` (one population, that is major population),
    - `one_major` (multiple populations with one major population),
    - `n_even_major` (n multiple populations, each major population has equivalent relative sample size),
    - `n_uneven_major` (n multiple populations, each major population has different relative sample size),
    - `empty_ld` (no LD structure provided - ex. molecular QTLs),
    - `no_relative_sample_size` (LD structure is provided, but without relative sample size.)
  - `majorPops`: a list of `LDPopulation` objects representing the major populations present in the LD structure. Each `LDPopulation` object contains:
    - `ldPopulation`: the ancestry value (ex. `nfe`, `afr`, etc.).
    - `relativeSampleSize`: the relative sample size for the population.

The reason for adding the `ldStructure` is to provide a more detailed information about the LD structure of major ancestries, the actual analysis makes the assumption that in the case of multiple major ancestries with even sample sizes, `nfe` is preferred over others, in case of uneven relative sample sizes, the largest major ancestry is preferred.


In [16]:
from manuscript_methods.ld_populations import LDPopulationName, LDPopulationStructure
from manuscript_methods.maf import AlleleFrequencies

ld_pop = LDPopulationStructure(f.col("ldPopulationStructure"))
major_ld_pop = ld_pop.major_population(default_major_pop=LDPopulationName.NFE)
major_ld_maf = AlleleFrequencies(f.col("alleleFrequencies")).ld_population_maf(major_ld_pop.ld_population)
major_ld_af = AlleleFrequencies(f.col("alleleFrequencies")).ld_population_af(major_ld_pop.ld_population)
ld_structure = ld_pop.ld_structure()

dataset = dataset.withColumns(
    {
        "majorLdPopulation": major_ld_pop.col,
        "majorLdPopulationMaf": major_ld_maf.col,
        "majorLdPopulationAf": major_ld_af.col,
        "majorLdPopulationsStructure": ld_structure,
    }
)

(
    dataset.select(
        "majorLdPopulation",
        "majorLdPopulationMaf",
        "majorLdPopulationAf",
        "majorLdPopulationsStructure",
    ).show(5, truncate=False)
)
(
    dataset.select(
        "majorLdPopulation",
        "majorLdPopulationMaf",
        "majorLdPopulationAf",
        "majorLdPopulationsStructure",
    ).printSchema()
)


26/04/09 09:10:11 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+-------------------------+---------------------------------+------------------------------+----------------------------------------+
|majorLdPopulation        |majorLdPopulationMaf             |majorLdPopulationAf           |majorLdPopulationsStructure             |
+-------------------------+---------------------------------+------------------------------+----------------------------------------+
|{nfe, 0.8790172333488588}|{0.40961283674370674, notFlipped}|{nfe_adj, 0.40961283674370674}|{one_major, [{nfe, 0.8790172333488588}]}|
|{nfe, 0.0}               |{0.40961283674370674, notFlipped}|{nfe_adj, 0.40961283674370674}|{empty_ld, []}                          |
|{nfe, 1.0}               |{0.40961283674370674, notFlipped}|{nfe_adj, 0.40961283674370674}|{only_major, [{nfe, 1.0}]}              |
|{nfe, 0.0}               |{0.40961283674370674, notFlipped}|{nfe_adj, 0.40961283674370674}|{empty_ld, []}                          |
|{nfe, 1.0}               |{0.40961283674370674, notFlipped}|{

## Variance explained by lead variant (Approximation)

The code below is used to calculate the PVE (Phenotypic Variance Explained) by the lead variant in the credible set.

The variance explained follows the simplified formula

${variance\;explained}=\chi^2 / n $

- The $\chi^2$ is calculated as **Inverse survival function** by using `scipy.stats.isf` function from lead variant $pValue$ (depicted as `pValueMantissa` and `pValueExponent`).
- The $n$ parameter is the number of samples derived from GWAS study description.

- In case where the `pValueExponent < 300` to avoid floating point errors we estimate $\chi^2$ statistic with $-log_{10}(pValue)$
- The $variance\;explained$ can be only calculated where the $n > 0$

* [Rosenberg MS. A generalized formula for converting chi-square tests to effect sizes for meta-analysis. PLoS One. 2010 Apr 7;5(4):e10059. doi: 10.1371/journal.pone.0010059. PMID: 20383281; PMCID: PMC2850938.](https://pmc.ncbi.nlm.nih.gov/articles/PMC2850938/)


In [17]:
from manuscript_methods.variant_statistics import PValueComponents, VariantStatistics

pval_components = PValueComponents(p_value_mantissa=f.col("pValueMantissa"), p_value_exponent=f.col("pValueExponent"))
n_samples = f.col("nSamples")
variant_stats = VariantStatistics.compute(pval_components, n_samples)

dataset = dataset.withColumns({"variantStatistics": variant_stats.col})
dataset.select("variantStatistics").show(5, truncate=False)
dataset.select("variantStatistics").printSchema()


/home/ss60/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
26/04/09 09:10:26 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:10:27 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:10:32 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use t

+----------------------------------------------------+
|variantStatistics                                   |
+----------------------------------------------------+
|{53.385590814285614, 2.741, -13, 0.5036376491913737}|
|{35.05757089255757, 3.201, -9, 0.1905302765899868}  |
|{41.60862990772693, 1.115, -10, 0.12921934753952463}|
|{28.025941710304732, 1.197, -7, 0.08492709609183252}|
|{33.63165583322489, 6.66, -9, 0.1419057208152949}   |
+----------------------------------------------------+
only showing top 5 rows

root
 |-- variantStatistics: struct (nullable = false)
 |    |-- chi2Stat: double (nullable = true)
 |    |-- pValueMantissa: float (nullable = true)
 |    |-- pValueExponent: integer (nullable = true)
 |    |-- ApproximatedVarianceExplained: double (nullable = true)



## Study statistics

The code below is used to combine and classify the cohort statistics from the _studyIndex_ dataset.

This includes:

- n_cases
- n_controls
- n_samples
- study_type
- trait
- trait_ids
- gene_id


In [18]:
from manuscript_methods.study_statistics import StudyStatistics

cohort_stat = StudyStatistics.compute(
    n_samples=f.col("nSamples"),
    n_cases=f.col("nCases"),
    n_controls=f.col("nControls"),
    trait=f.col("traitFromSource"),
    study_type=f.col("studyType"),
    is_trans_pqtl=f.col("isTransQtl"),
    gene_id=f.col("geneId"),
)

dataset = dataset.withColumns({"studyStatistics": cohort_stat.col})
dataset.select("studyStatistics").show(5, truncate=False)
dataset.select("studyStatistics").printSchema()


26/04/09 09:10:44 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:10:48 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+--------------------------------------------------------------------------------------------------------+
|studyStatistics                                                                                         |
+--------------------------------------------------------------------------------------------------------+
|{NULL, NULL, 106, ENSG00000141698, sceqtl, quantitative, ENSG00000141698}                               |
|{NULL, NULL, 184, ENSG00000141698.grp_2.contained.ENST00000393910, tuqtl, quantitative, ENSG00000141698}|
|{NULL, NULL, 322, 17:41813279:41818089:clu_40451_-, sqtl, quantitative, ENSG00000141756}                |
|{NULL, NULL, 330, ENSG00000141756, eqtl, quantitative, ENSG00000141756}                                 |
|{NULL, NULL, 237, 17:41817203:41818089:clu_21452_+, sqtl, quantitative, ENSG00000141756}                |
+--------------------------------------------------------------------------------------------------------+
only showing top 5 rows

root
 |-- st

## Rescaling of the marginal effect size

Due to the fact that the `beta` reported by the credible sets lead variant can be of unknown origin ($\mu$ from SuSiE or beta from summary statistics) or scale, we want to rescale the marginal effect size from the $\chi^2$ statistic.

The rescaling of the marginal effect size is done via two formulas depending on trait being **quantitative** or **binary**.

### Procedure overview

1. Compute the $\chi^2$ statistic from the $p-value$ of the lead variant.
2. Estimate the model used for beta estimation based on the trait type (binary or quantitative).
3. Compute the $z-score$ from the $\chi^2$ statistic value.
4. Compute the standard error ($se$) based on the model type and MAF of the lead variant.
5. Compute the rescaled marginal effect size.

### $\chi^2$ computation

The $\chi2^2$ is computed as the inverse survival function of the $pValue$ of the lead variant using `scipy.stats.isf` function. This step is done in the VariantStatistics paragraph.

### Trait type estimation

Estimation of the trait type is done on the basis of availability of reported `nCases` and `nControls` fields in the study description. This step is done in the StudyStatistics paragraph.

- In case both fields are non empty and non zero we assume _binary trait_
- In case cases are zero or are not reported we assume _quantitative trait_

### Calculation of the $Z score$

The $Z$ is computed as follows:

$$Z = \frac{\beta}{|{\beta}|} \cdot \sqrt{\chi^2}$$

Where

- $\beta$ - _standardised beta reported from in the summary statistics_
- $\frac{\beta}{|{\beta}|}$ is used to find the direction of the effect.
- In case when $\beta$ was not reported we assumed the $\frac{\beta}{|{\beta}|}$ to be unknown

### Calculation of the standard error ($se$)

In both cases we estimate the marginal effect size $estimated\;\beta$ with following formula
$$estimated\;\beta = Z \cdot se$$

#### Binary trait marginal effect size estimation

$$se = \frac{1}{\sqrt{(varG \cdot prev \cdot (1 - prev))}}$$

- $varG = 2  \cdot f \cdot (1 - f)$ - _component of genetic variance_ - the original is $var_{G} = 2\beta^2f(1 - f)$
- $f$ - _Minor Allele Frequency of lead variant_
- $prev = \frac{nCases}{nSamples}$ - _Trait prevelance_

#### Quantative trait marginal effect size estimation

$$se = \frac{1}{\sqrt{n \cdot varG}}$$

- $varG = 2  \cdot f \cdot (1 - f)$
- $f$ - _Minor Allele Frequency of lead variant_

The $\chi^2$ was esteimated as described in `variance Explained` calculation.


In [19]:
from manuscript_methods.maf import MinorAlleleFrequency, PopulationFrequency
from manuscript_methods.rescaled_beta import RescaledStatistics

rescaled_stats = RescaledStatistics.compute(
    beta=f.col("beta"),
    chi2_stat=VariantStatistics(f.col("variantStatistics")).chi2_stat,
    trait_class=StudyStatistics(f.col("studyStatistics")).trait_class,
    af=PopulationFrequency(f.col("majorLdPopulationAf")).allele_frequency,
    maf=MinorAlleleFrequency(f.col("majorLdPopulationMaf")).value,
    n_cases=StudyStatistics(f.col("studyStatistics")).n_cases,
    n_samples=StudyStatistics(f.col("studyStatistics")).n_samples,
)

dataset = dataset.withColumns({"rescaledStatistics": rescaled_stats.col})

dataset.select("rescaledStatistics").show(5, truncate=False)
dataset.select("rescaledStatistics").printSchema()

# Check for empty rescaled beta

# NOTE:
# Reasons for null rescaled beta could be:
# - missing beta in the summary statistics
# - af == 0.0

(
    dataset.select(
        f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
        f.col("rescaledStatistics.directionOfEffect").alias("directionOfEffect"),
        "beta",
        f.col("rescaledStatistics.absZScore").alias("absZScore"),
        f.col("rescaledStatistics.minorAlleleEstimatedBeta").alias("minorAlleleEstimatedBeta"),
        f.col("majorLdPopulationMaf.value").alias("maf"),
    )
).show(5, truncate=False)

(
    dataset.select(
        "rescaledStatistics",
        "beta",
        f.col("majorLdPopulationMaf.value").alias("maf"),
        f.col("majorLdPopulationAf.alleleFrequency").alias("af"),
    )
    .filter(f.col("rescaledStatistics").getField("absEstimatedBeta").isNull())
    .select("rescaledStatistics.absEstimatedBeta", "beta", "maf", "af")
    .show(5, truncate=False)
)


26/04/09 09:11:03 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+-------------------------------------------------------------------------------------------------------------------+
|rescaledStatistics                                                                                                 |
+-------------------------------------------------------------------------------------------------------------------+
|{1, 18.45833333179044, 0.49178813385915704, 0.0, 0.002395137352595494, 0.04421024362962971, -0.04421024362962971}  |
|{-1, 5.958333330564855, 0.49178813385915704, 0.0, 0.002390300346715964, 0.014242206225898457, 0.014242206225898457}|
|{-1, 6.195962273807517, 0.4887360593415791, NULL, 0.09101528470975516, 0.5639272704014932, -0.5639272704014932}    |
|{-1, 5.813883230285689, 0.4887360593415791, NULL, 0.11718436591290564, 0.681296219832704, -0.681296219832704}      |
|{1, 6.879792035887576, 0.089859992550242, NULL, 0.280935909406388, 1.9327806321289018, 1.9327806321289018}         |
+-------------------------------------------------------

26/04/09 09:11:24 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:11:26 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+--------------------+-----------------+---------------------+-----------------+------------------------+-------------------+
|absEstimatedBeta    |directionOfEffect|beta                 |absZScore        |minorAlleleEstimatedBeta|maf                |
+--------------------+-----------------+---------------------+-----------------+------------------------+-------------------+
|0.04421024362962971 |1                |0.030208797815470614 |18.45833333179044|-0.04421024362962971    |0.4359224448779334 |
|0.014242206225898457|-1               |-0.009145260427008407|5.958333330564855|0.014242206225898457    |0.4359224448779334 |
|0.5639272704014932  |-1               |-0.351583            |6.195962273807517|-0.5639272704014932     |0.4249535455253851 |
|0.681296219832704   |-1               |-0.663637            |5.813883230285689|-0.681296219832704      |0.4249535455253851 |
|1.9327806321289018  |1                |1.05413              |6.879792035887576|1.9327806321289018      |0.04715344350

26/04/09 09:11:45 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+----------------+---------------------+------------------+------------------+
|absEstimatedBeta|beta                 |maf               |af                |
+----------------+---------------------+------------------+------------------+
|NULL            |2162.0               |0.0               |0.0               |
|NULL            |20.726               |0.0               |0.0               |
|NULL            |0.13976194237515863  |0.4748062015503876|0.4748062015503876|
|NULL            |-0.03664046667911124 |0.0               |0.0               |
|NULL            |-0.031037430576473315|0.0               |0.0               |
+----------------+---------------------+------------------+------------------+
only showing top 5 rows



## VEP consequence extraction

The VEP consequence extraction is done with two manners:

1. Looking for the **direct most severe consequence** for the lead variant. This is done by linking the lead variant to the gene with most severe consequence in within the +/- 500kb region of the lead variant. - `mostSevereConsequence`
2. Looking for the consequence linked to the molecular QTL egene
   - If the `geneId` (egene) is found in transcript annotations (**in-gene effect**),
   - If the `geneId` (egene) is not found in the transcript consequences, we expect that the gene within the +/- 500kb region of the lead variant, which has the most severe consequence to have distal effect (**out-gene effect**) on the egene. - `mostSevereConsequenceForMolecularTrait`


In [20]:
from manuscript_methods.study_statistics import StudyStatistics
from manuscript_methods.tc import LeadVariantConsequences, TranscriptConsequences

tc = TranscriptConsequences(f.col("transcriptConsequences"))
sstats = StudyStatistics(f.col("studyStatistics"))
lc = LeadVariantConsequences.compute(tc, sstats)


dataset = dataset.withColumn(lc.name, lc.col)
dataset.select(lc.name).show(5, truncate=False)
dataset.select(lc.name).printSchema()


26/04/09 09:12:13 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|leadVariantConsequence                                                                                                                                                                                                                                                                                                                                                                       |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Locus statistics

The locus statistics gets:

- Posterior Probability of the lead variant
- Locus length
- Locus size
- Locus start - (start of the first variant in the locus)
- Locus end - (end of the last variant in the locus)

All statistics are derived from the **studyLocus** dataset `locus` object.


In [21]:
from manuscript_methods.locus_statistics import LocusStatistics

ls = LocusStatistics.compute(locus=f.col("locus"), lead_variant=f.col("variantId"))
dataset = dataset.withColumn(ls.name, ls.col)
dataset.select(ls.name).show(5, truncate=False)
dataset.select(f"{ls.name}.*").printSchema()


26/04/09 09:14:47 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/09 09:14:48 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+-------------------------------------------------------+
|locusStatistics                                        |
+-------------------------------------------------------+
|{114, 115355, 41818912, 41934267, 0.056167909330217994}|
|{89, 134578, 41837233, 41971811, 0.20658209386018897}  |
|{5, 63985, 41900339, 41964324, 0.29215398192206327}    |
|{6, 129249, 41816944, 41946193, 0.24757336534740973}   |
|{4, 45854, 41900339, 41946193, 0.30917649298786726}    |
+-------------------------------------------------------+
only showing top 5 rows

root
 |-- locusSize: integer (nullable = false)
 |-- locusLength: integer (nullable = true)
 |-- locusStart: integer (nullable = true)
 |-- locusEnd: integer (nullable = true)
 |-- leadVariantPIP: double (nullable = true)



## Variant type for interval joins

Compute the variant type and effective length of Indels to use them downstream for interval joins


In [22]:
from manuscript_methods.variant_type import Variant

dataset = dataset.withColumn(
    "variant",
    Variant.compute(f.col("chromosome"), f.col("position"), f.col("referenceAllele"), f.col("alternateAllele")).col,
)

dataset.select("variant").show(5, truncate=False)
dataset.select("variant").printSchema()


26/04/09 09:15:03 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+----------------------------------------+
|variant                                 |
+----------------------------------------+
|{10, 62187275, 62187275, SNV, C, T, 0}  |
|{2, 101031031, 101031032, INS, T, TC, 1}|
|{2, 101031031, 101031032, INS, T, TC, 1}|
|{2, 101031031, 101031032, INS, T, TC, 1}|
|{7, 11061679, 11061679, SNV, G, C, 0}   |
+----------------------------------------+
only showing top 5 rows

root
 |-- variant: struct (nullable = false)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: integer (nullable = true)
 |    |-- end: integer (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- ref: string (nullable = true)
 |    |-- alt: string (nullable = true)
 |    |-- length: integer (nullable = true)



## Final dataset contract


In [23]:
dataset = dataset.select(
    f.col("variantId"),
    f.col("variant"),
    f.col("studyLocusId"),
    f.col("studyId"),
    f.col("geneId"),
    f.col("diseaseIds"),
    f.col("biosampleId"),
    f.col("beta").alias("originalBeta"),
    f.col("standardError").alias("originalStandardError"),
    f.col("locusStatistics"),
    f.col("finemappingMethod"),
    f.col("isTransQtl"),
    f.col("variantEffect"),
    f.col("majorLdPopulation"),
    f.col("majorLdPopulationMaf"),
    f.col("majorLdPopulationAf"),
    f.col("variantStatistics"),
    f.col("studyStatistics"),
    f.col("rescaledStatistics"),
    f.col("leadVariantConsequence"),
    f.col("traitFromSourceMappedIds"),
)


### Save the dataset to parquet


In [ ]:
dataset.repartition(50).write.mode("overwrite").parquet(output_dataset_path)


26/04/09 09:15:08 ERROR YarnScheduler: Lost executor 2 on ot-genetics-dev-000-ss60-w-1.europe-west1-b.c.open-targets-genetics-dev.internal: Executor decommission finished: spark scale down (31.0s) - Migration: 33/33 blocks (839.9 MiB/4.1 GiB), 16 deleted
26/04/09 09:15:08 WARN BlockManagerMaster: Failed to remove broadcast 82 with removeFromMaster = true - org.apache.spark.SparkException: Could not find BlockManagerEndpoint1.
	at org.apache.spark.rpc.netty.Dispatcher.postMessage(Dispatcher.scala:178)
	at org.apache.spark.rpc.netty.Dispatcher.postRemoteMessage(Dispatcher.scala:136)
	at org.apache.spark.rpc.netty.NettyRpcHandler.receive(NettyRpcEnv.scala:683)
	at org.apache.spark.network.server.TransportRequestHandler.processRpcRequest(TransportRequestHandler.java:163)
	at org.apache.spark.network.server.TransportRequestHandler.handle(TransportRequestHandler.java:109)
	at org.apache.spark.network.server.TransportChannelHandler.channelRead0(TransportChannelHandler.java:140)
	at org.apache

26/04/09 09:24:51 ERROR YarnScheduler: Lost executor 8 on ot-genetics-dev-000-ss60-sw-wkr2.europe-west1-b.c.open-targets-genetics-dev.internal: Executor decommission finished: spark scale down (31.1s) - Migration: 146/146 blocks (2.6 GiB/2.6 GiB), 0 deleted
26/04/09 09:24:54 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 8 for reason Executor for container container_1775724357121_0001_01_000008 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/09 09:25:17 ERROR YarnScheduler: Lost executor 3 on ot-genetics-dev-000-ss60-w-0.europe-west1-b.c.open-targets-genetics-dev.internal: Executor decommission finished: spark scale down (61.0s) - Migration: 107/107 blocks (6.0 GiB/6.0 GiB), 0 deleted
26/04/09 09:25:17 ERROR YarnScheduler: Lost executor 6 on ot-genetics-dev-000-ss60-w-1.europe-west1-b.c.open-targets-genetics-dev.internal: Executor decommission finished: spark scale down (61.0s) - Migration: 1

### Save the schema


In [ ]:
# Write only when running locally
if output_dataset_schema.startswith("gs://"):
    pass
else:
    with open(output_dataset_schema, "w") as fp:
        json.dump(json.loads(dataset.schema.json()), fp, indent=2)


In [ ]:
print(f"Dataset with {dataset.count()} rows saved to {output_dataset_path}")


Dataset with 2833758 rows saved to ../../data/intermediate_files/lead_variant_effect
